# Bonus B4 solution, notebook 2: a refund under the limit

The worked answer to the challenge of `b4b_crewai_at_the_desk.ipynb`. The setup repeats the notebook's lookups,
agents, type and guardrail, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
# CrewAI reads these when it is imported, so they come first
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"      # no usage statistics sent to CrewAI
os.environ["CREWAI_TRACING_ENABLED"] = "false"       # no traces sent to CrewAI's hosted platform
os.environ["CREWAI_DISABLE_VERSION_CHECK"] = "true"  # no check for a newer CrewAI on every start
os.environ["NO_COLOR"] = "1"                         # the crew's log prints without colours

# CrewAI's own files (the last run's task outputs) go to this lab's outputs/ folder, which git ignores
(Path.cwd() / "outputs").mkdir(exist_ok=True)
os.environ["CREWAI_STORAGE_DIR"] = str(Path.cwd() / "outputs")

In [3]:
import json
import sqlite3
import threading
from contextlib import closing
from typing import Literal
from pydantic import BaseModel
from crewai import LLM, Agent, Task, Crew
from crewai.tools import tool

DB = ROOT / "data" / "chinook.db"


@tool("find_customer")
def find_customer(name: str) -> str:
    """Find a shop customer by full name. Returns the customer_id, name and country."""
    print(f"🔎 TOOL CALLED: find_customer({name}) in {threading.current_thread().name}")
    with closing(sqlite3.connect(f"file:{DB}?mode=ro", uri=True)) as con:
        rows = con.execute(
            "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
            "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    if not rows:
        return f"no match for customer {name}"
    return json.dumps([{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows], ensure_ascii=False)


@tool("get_invoice")
def get_invoice(invoice_id: int) -> str:
    """Look up one invoice by its number. Returns the customer_id, the date and the total in US dollars."""
    print(f"🔎 TOOL CALLED: get_invoice({invoice_id}) in {threading.current_thread().name}")
    with closing(sqlite3.connect(f"file:{DB}?mode=ro", uri=True)) as con:
        rows = con.execute(
            "SELECT InvoiceId, CustomerId, date(InvoiceDate), Total FROM invoices WHERE InvoiceId = ?",
            (invoice_id,)).fetchall()
    if not rows:
        return f"no match for invoice {invoice_id}"
    invoice_id, customer_id, day, total = rows[0]
    return json.dumps({"invoice_id": invoice_id, "customer_id": customer_id, "date": day, "total": total})


class RefundAdvice(BaseModel):
    customer_name: str
    invoice_id: int
    amount: float
    action: Literal["refund", "human_review"]
    reason: str


REFUND_LIMIT = 10.00


def check_advice(output):
    advice = output.pydantic
    if advice.action == "refund" and advice.amount > REFUND_LIMIT:
        print(f"❌ refused: refund of ${advice.amount:.2f}")
        return (False, f"A refund above ${REFUND_LIMIT:.2f} needs a person: the action must be human_review.")
    print(f"✅ accepted: {advice.action}, ${advice.amount:.2f}")
    return (True, output.raw)


def new_llm():
    return LLM(model=f"openai/{MODEL}", api="responses", reasoning_effort=EFFORT, timeout=60)


def new_clerk():
    return Agent(
        role="Billing clerk",
        goal="Look up customers and invoices in the shop's database and report exactly what it holds.",
        backstory="You work at the support desk of the Chinook music shop.",
        tools=[find_customer, get_invoice],
        llm=new_llm(),
    )


def new_adviser():
    return Agent(
        role="Refund adviser",
        goal="Advise the support rep on what to do with a refund request.",
        backstory="You work at the support desk of the Chinook music shop. Duplicate charges are refunded in full.",
        llm=new_llm(),
    )


LOOKUP = "A support rep passes on this request: {request}\nLook up the customer and the invoice."
FACTS = "The customer's id and name, and the invoice's number, date and total."
ADVISE = "A support rep passes on this request: {request}\nAdvise the rep what to do."

## Invoice 284 through Part D's crew

Only the request changes. The crew, the type and the guardrail are Part D's.

In [4]:
REQUEST = "Puja Srivastava says invoice 284 was charged to her card twice and asks for her money back."

clerk, adviser = new_clerk(), new_adviser()
lookup_task = Task(description=LOOKUP, expected_output=FACTS, agent=clerk)
advice_task = Task(description=ADVISE, expected_output="The advice.", agent=adviser,
                   context=[lookup_task], output_pydantic=RefundAdvice, guardrail=check_advice)

result = await Crew(agents=[clerk, adviser], tasks=[lookup_task, advice_task]).akickoff(inputs={"request": REQUEST})

print()
for field, value in result.pydantic.model_dump().items():
    print(f"  {field}: {value}")
print("model requests:", result.token_usage.successful_requests)

🔎 TOOL CALLED: find_customer(Puja Srivastava) in ThreadPoolExecutor-3_0
🔎 TOOL CALLED: get_invoice(284) in ThreadPoolExecutor-3_1


✅ accepted: refund, $8.91

  customer_name: Puja Srivastava
  invoice_id: 284
  amount: 8.91
  action: refund
  reason: Invoice 284 was charged twice. Duplicate charges are refunded in full; refund $8.91.
model requests: 3


The two lookups ran in two threads, as in Part D. Invoice 284 is Puja's and comes to $8.91, under the $10 limit, so
the adviser's refund passed `check_advice` at the first attempt: one ✅ line and **3 model requests**, one fewer than
Part D's run, which needed a second attempt. The reason repeats the desk's habit (*duplicate charges are refunded in
full*) and does not mention the limit: the adviser's prompt never carried it. The rule is applied by our code, and the
advice does not need to explain it.